In [ ]:
import os
import shutil
import tarfile
import glob
from pathlib import Path

# ==========================================
# 1. Configuration (설정값 관리)
# ==========================================
class Config:
    # 데이터셋 정보
    DATASET_ID = "004"
    DATASET_NAME = "Dataset004_Hippocampus"
    CONFIGURATION = "3d_fullres"
    FOLD = 0

    # 경로 설정
    BASE_DIR = Path("/content")
    DRIVE_DIR = BASE_DIR / "drive/MyDrive/nnUNet_Hippocampus_Final_v1"
    RAW_DATA_TAR = BASE_DIR / "drive/MyDrive/kvar/Task04_Hippocampus.tar"

    # nnU-Net 필수 환경 변수 경로
    NNUNET_RAW = BASE_DIR / "nnUNet_raw"
    NNUNET_PREPROCESSED = BASE_DIR / "nnUNet_preprocessed"
    NNUNET_RESULTS = BASE_DIR / "nnUNet_results"

    # 모델 저장 경로
    MODEL_BASE_PATH = NNUNET_RESULTS / DATASET_NAME / f"nnUNetTrainer__nnUNetPlans__{CONFIGURATION}"
    FOLD_PATH = MODEL_BASE_PATH / f"fold_{FOLD}"

    # 출력 경로
    EXTRACT_PATH = BASE_DIR / "test_data"
    OUTPUT_DIR = BASE_DIR / "inference_results"

def setup_environment():
    """nnU-Net 환경 변수 설정 및 디렉토리 생성"""
    print("🌐 환경 설정 중...")
    os.environ['nnUNet_raw'] = str(Config.NNUNET_RAW)
    os.environ['nnUNet_preprocessed'] = str(Config.NNUNET_PREPROCESSED)
    os.environ['nnUNet_results'] = str(Config.NNUNET_RESULTS)

    Config.FOLD_PATH.mkdir(parents=True, exist_ok=True)
    Config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def restore_model_files():
    """드라이브에서 학습된 모델 파일 복구"""
    print("📂 모델 파일 복구 시작...")
    files_to_copy = [
        (Config.DRIVE_DIR / "dataset.json", Config.MODEL_BASE_PATH),
        (Config.DRIVE_DIR / "plans.json", Config.MODEL_BASE_PATH),
        (Config.DRIVE_DIR / "checkpoint_best.pth", Config.FOLD_PATH),
    ]

    for src, dst in files_to_copy:
        if src.exists():
            shutil.copy(src, dst)
            print(f"✅ {src.name} -> {dst}")
        else:
            print(f"⚠️ 경고: {src} 파일을 찾을 수 없습니다.")

def prepare_input_data():
    """데이터 압축 해제 및 전처리 (파일명 정규화 및 노이즈 제거)"""
    print("📦 데이터 준비 중...")

    # 압축 해제
    if not Config.EXTRACT_PATH.exists():
        with tarfile.open(Config.RAW_DATA_TAR, 'r') as tar:
            tar.extractall(path=Config.EXTRACT_PATH)

    images_dir = Config.EXTRACT_PATH / 'Task04_Hippocampus/imagesTr'

    # 1. macOS 유령 파일(._) 삭제
    ghost_files = glob.glob(str(images_dir / "._*"))
    for gf in ghost_files:
        os.remove(gf)
    if ghost_files: print(f"🧹 {len(ghost_files)}개의 유령 파일 삭제 완료.")

    # 2. 파일명 보정 (_0000.nii.gz)
    for f_path in images_dir.glob("*.nii.gz"):
        if "_0000" not in f_path.name:
            new_name = f_path.name.replace(".nii.gz", "_0000.nii.gz")
            f_path.rename(images_dir / new_name)

    print(f"✅ 데이터셋 준비 완료: {images_dir}")
    return images_dir

def run_inference(input_dir):
    """nnU-Net 추론 실행"""
    print("\n🤖 AI 추론 시작...")

    # f-string을 활용한 커맨드 구성
    cmd = (
        f"nnUNetv2_predict -i {input_dir} "
        f"-o {Config.OUTPUT_DIR} "
        f"-d {Config.DATASET_ID} "
        f"-c {Config.CONFIGURATION} "
        f"-f {Config.FOLD} "
        f"-chk checkpoint_best.pth"
    )

    os.system(cmd)
    print(f"🚀 추론 완료! 결과 저장 위치: {Config.OUTPUT_DIR}")

# ==========================================
# 실행부
# ==========================================
if __name__ == "__main__":
    # 라이브러리 설치는 터미널이나 별도 셀에서 권장하지만,
    # 필요한 경우 여기에 포함: os.system("pip install nnunetv2")

    setup_environment()
    restore_model_files()
    target_images = prepare_input_data()
    run_inference(target_images)